In [ ]:
import os

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.functional as F
import torchvision
import torchvision.transforms as transforms


In [ ]:
print(f"Torch Version : {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version : {torch.version.cuda}")
    print(f"Device Name : {torch.cuda.get_device_name()}")

In [ ]:
device = "cuda" if torch.cuda.is_available else "cpu"
print(device)

In [ ]:
sample_dir = "sample"
if not os.path.exists(sample_dir):
    os.makedirs(sample_dir)
    print(f"Created Directory : {sample_dir}")

In [ ]:
latent_size = 64
hidden_size = 256
image_size = 784
num_epochs = 100
batch_size = 100

<!-- ### Data Preperation and Loading -->

In [ ]:
def load_mnist_data(batch_size : int) -> torch.utils.data.DataLoader :

    # Data Preprocessing Pipeline
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            mean = [0.5],
            std = [0.5],
        )
    ])

    # Load MNIST training dataset with transformations
    mnist = torchvision.datasets.FashionMNIST(
        root = "./data/",
        train = True,
        transform = transform,
        download = True,
    )

    # Create DataLoader for effecient batch processing
    data_loader = torch.utils.data.DataLoader(
        dataset = mnist,
        batch_size = batch_size,
        shuffle = True,
    )

    return data_loader



In [ ]:
data_loader = load_mnist_data(batch_size)

<!-- ### Model Architecture -->

In [ ]:
class Generator(nn.Module):

    def __init__(self , latent_size , hidden_size , image_size):

        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(latent_size , hidden_size),
            nn.LeakyReLU(0.2),

            nn.Linear(hidden_size , hidden_size),
            nn.LeakyReLU(0.2),

            nn.Linear(hidden_size , image_size),
            nn.Tanh(),
        )

    
    def forward(self , z):
        return self.model(z)


In [ ]:
generator = Generator(latent_size , hidden_size , image_size)

In [ ]:
class Discriminator(nn.Module):

    def __init__(self , image_size , hidden_size):
        super().__init__()

        self.model = nn.Sequential(

            nn.Linear(image_size , hidden_size),
            nn.LeakyReLU(0.2),

            nn.Linear(hidden_size , hidden_size),
            nn.LeakyReLU(0.2),

            nn.Linear(hidden_size , 1),
            nn.Sigmoid(),

        )

    def forward(self , x):
        return self.model(x)


In [ ]:
discriminator = Discriminator(image_size , hidden_size).to(device)

In [ ]:
test_image = torch.randn(1 , image_size).to(device)
test_classification = discriminator(test_image)

<!-- ### Optimizers -->

In [ ]:
criterion = nn.BCELoss()

d_optimizer = torch.optim.Adam(discriminator.parameters() , lr = 0.0002)
g_optimizer = torch.optim.Adam(generator.parameters() , lr = 0.0002)

In [ ]:
def denorm(x):    
    out = (x+1)/2
    return out.clamp(0,1)

def reset_grad():
    d_optimizer.zero_grad()
    g_optimizer.zero_grad()



<!-- ### Training -->

In [ ]:
total_step = len(data_loader)
d_losses = []
g_losses = []

for epoch in range(1 , num_epochs+1):

    epoch_d_loss = 0
    epoch_g_loss = 0

    for i , (images , _) in enumerate(data_loader):

        images = images.reshape(batch_size , -1).to(device)

        real_labels = torch.ones(batch_size , 1).to(device)
        fake_labels = torch.zeros(batch_size , 1).to(device)

        outputs = discriminator(images)
        d_loss_real = criterion(outputs , real_labels)
        real_score = outputs


        z = torch.randn(batch_size , latent_size).to(device)
        fake_images = generator(z)


        outputs = discriminator(fake_images.detach())
        d_loss_fake = criterion(outputs , fake_labels)
        fake_score = outputs



        d_loss = d_loss_real + d_loss_fake
        reset_grad()
        d_loss.backward()
        d_optimizer.step()


        z = torch.randn(batch_size , latent_size).to(device)
        fake_images = generator(z)


        outputs = discriminator(fake_images)
        g_loss = criterion(outputs , real_labels)

        reset_grad()
        g_loss.backward()
        g_optimizer.step()



        epoch_d_loss += d_loss.item()
        epoch_g_loss += g_loss.item()
        